Week 1 assignment
Author : Yasine Deghaies

This project combines ebay scraped data + LLM insights, which reflects the core idea behind RAG -  uniting private external data sources with LLM knowledge.

This is a simple and yet still evolving project to identify listings with highest flip potential (based on historical data)

Features
	•	Scrapes eBay sold listings
	•	Generate AI summary of market trends
	•	Top 3 listings with highest flip potential (based on historical data)


In [84]:
#Import Dependencies
import requests
from bs4 import BeautifulSoup
import pandas as pd
from openai import OpenAI
import os
from dotenv import load_dotenv

In [85]:
# Define the eBay filters dictionary
ebay_filters = {
    "item_conditions": {
        "New": 1000,
        "Open box": 1500,
        "Used": 3000,
        "Certified Refurbished": 2000,
        "Excellent - Refurbished": 2500,
        "Very Good": 3000,
        "Good": 4000,
        "For Parts or Not Working": 7000
    },
    "item_locations": {
        "Domestic": 1,
        "International": 2,
        "Continent": 3,
    },
    "directories": {
        "No Directory": 0,
        "Consumer Electronics": 9355,
        "Clothing, Shoes & Accessories": 11450,
        "Health & Beauty": 26395,
        "Home & Garden": 11700,
        "Sporting Goods": 382,
        "Toys & Hobbies": 220,
        "Books": 267,
        "Video Games & Consoles": 1249,
        "Collectibles": 1,
        "Business & Industrial": 12576,
        "Automotive": 6000, 
    },
    "categories": {
        "No Category": 0,
        "Cell Phones & Smartphones": 9355,
        "Laptops & Netbooks": 175673,
        "Watches": 31387,
        "Furniture": 3197,
        "Action Figures": 2605,
        "Jewelry & Watches": 281,
        "Cameras & Photo": 625,
        "Pet Supplies": 1281,
        "Crafts": 14339,
        "Computers/Tablets & Networking": 58058,
        "Cars & Trucks": 6001,  
        "Motorcycles": 6024,  
        "Car & Truck Parts": 6030,  
        "Motorcycle Parts": 10063,  
        "Automotive Tools & Supplies": 34998,  
    },
    "sort_order": {
        "Best Match": 12,
        "Time: ending soonest": 1,
        "Time: newly listed": 10,
        "Price + Shipping: lowest first": 15,
        "Price + Shipping: highest first": 16,
        "Distance: nearest first": 7
    }
}


In [ ]:
# Define the base URL for the eBay search
url = "https://www.ebay.com/sch/i.html"

# Define the query parameters for the search request
params = {
    '_from': 'R40',
    '_nkw': 'iphone 13',
    'LH_ItemCondition': ebay_filters["item_conditions"]["Used"],  # Item condition; 'New'.
    'LH_PrefLoc': ebay_filters["item_locations"]["International"],
    '_udlo': '200',  # Minimum price.
    '_udhi': '400',  # Maximum price.
    '_dcat': ebay_filters["directories"]["No Directory"],  # Filter by directory ID; "Consumer Electronics".
    '_sacat': ebay_filters["categories"]["No Category"],  # Filter by category ID; "Cell Phones & Smartphones".
    '_sop': ebay_filters["sort_order"]["Time: newly listed"],  # Sort by "Time: newly listed"
    'LH_Sold': '1',  # Only sold listings (='1').
    'LH_Complete': '1',  # Only completed listings (='1').
    'LH_BIN': '0',  # Only Buy It Now listings (='1').
    'LH_Auction': '1',  # Only Auction Listings (='1').
    'LH_BO': '0',  # Only listings that accept offers (='1').
    'LH_FS': '0',  # Only Free Shipping listings (='1').
    '_ipg': '240',  # Number of items per page (='1'), Max is 240.
    'rt': 'nc'  # Result type; 'nc' indicates no cache to ensure the search results are fresh. 
    
}


In [87]:
# Create a prepared request to inspect the full URL
request  =requests.Request('GET', url, params=params)
prepared_request = request.prepare()
print(prepared_request.url)


https://www.ebay.de/sch/i.html?_from=R40&_nkw=iphone+13&LH_ItemCondition=3000&LH_PrefLoc=2&_udlo=200&_udhi=400&_dcat=0&_sacat=0&_sop=10&LH_Sold=1&LH_Complete=1&LH_BIN=0&LH_Auction=1&LH_BO=0&LH_FS=0&_ipg=240&rt=nc


In [88]:
# Initialize variables
page_number = 0
items_list = []

# Loop over pages
while True:
    
    page_number += 1
    
    print(f'Scraping page: {page_number}')
          
    params['_pgn'] = page_number
    
    # Send GET request to eBay with the defined parameters
    response  = requests.get(url, params=params)
    html_content = response.text # Get the HTML content of the page
    
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(html_content, 'html.parser')
          
    # Find the next button
    next_button = soup.find('button', class_="pagination__next icon-btn")

    # Extract items
    items = soup.find_all("div", class_="su-card-container su-card-container--horizontal")
    # Extract Listings
    for item in items [2:]:
        title = item.find("div", class_="s-card__title").text
        
        price_tag = item.find("span", class_="su-styled-text positive bold large-1 s-card__price")
        price = price_tag.get_text(strip=True) if price_tag else None    
            
        link = item.find("a", class_="s-card__link")['href'].split("?")[0]
        img_tag = item.find("img")
        image_url = img_tag.get("src", "No image URL") if img_tag else "No image URL"
        
        # Define each item as a dictionary
        item_dict = {
            'Title': title,
            'Price': price,
            'Link': link,
            'Image Link': image_url
        }

        # Append the dictionary to the list
        items_list.append(item_dict)
        
    if next_button and next_button.get('aria-disabled') == 'true':
        # Check if the next button is disabled
        print('No more pages pages to scrape')
        break


Scraping page: 1
Scraping page: 2
Scraping page: 3
Scraping page: 4
Scraping page: 5
Scraping page: 6
Scraping page: 7
Scraping page: 8
Scraping page: 9
Scraping page: 10
Scraping page: 11
Scraping page: 12
Scraping page: 13
Scraping page: 14
Scraping page: 15
Scraping page: 16
Scraping page: 17
Scraping page: 18
Scraping page: 19
Scraping page: 20
Scraping page: 21
Scraping page: 22
Scraping page: 23
Scraping page: 24
Scraping page: 25
Scraping page: 26
Scraping page: 27
Scraping page: 28
Scraping page: 29
Scraping page: 30
Scraping page: 31
Scraping page: 32
Scraping page: 33
Scraping page: 34
Scraping page: 35
Scraping page: 36
Scraping page: 37
Scraping page: 38
Scraping page: 39
Scraping page: 40
Scraping page: 41
Scraping page: 42
Scraping page: 43
Scraping page: 44
Scraping page: 45
Scraping page: 46
Scraping page: 47
Scraping page: 48
Scraping page: 49
Scraping page: 50
Scraping page: 51
Scraping page: 52
Scraping page: 53
Scraping page: 54
Scraping page: 55
Scraping page: 56
S

In [89]:
len(items_list)

316

In [90]:
# Turn the list into a Pandas DataFrame
items_df = pd.DataFrame(items_list)

In [91]:
items_df

,Title,Price,Link,Image Link
0,Apple iPhone 13 Pro Max - 1TB - Silber (Ohne S...,"EUR 345,00",https://www.ebay.de/itm/117114615000,https://i.ebayimg.com/images/g/B4AAAeSwhFFpyRG...
1,Apple iPhone 15 - Pink 128 GB - ohne Simlock,"EUR 310,00",https://www.ebay.de/itm/257439157839,https://i.ebayimg.com/images/g/o0MAAeSwmFBpxRR...
2,Neues AngebotApple iPhone 13 mini - 256GB - Weiß,"EUR 200,00",https://www.ebay.de/itm/188246394806,https://i.ebayimg.com/images/g/IdwAAeSwLM5p0hx...
3,iPhone 13 Pro 128GB Alpingrün,"EUR 242,00",https://www.ebay.de/itm/157794907669,https://i.ebayimg.com/images/g/bWMAAeSwN~9pyP1...
4,Apple iPhone 13 Pro A2638 - 128GB - Sierrablau...,"EUR 256,00",https://www.ebay.de/itm/389835127033,https://i.ebayimg.com/images/g/f98AAeSwmGVpzj1...
...,...,...,...,...
311,Apple iPhone 13 - 512GB - Starlight (Ohne Siml...,"EUR 243,83",https://www.ebay.de/itm/358064847215,https://i.ebayimg.com/images/g/7TsAAeSwsdxpUrI...
312,Apple iPhone 13 Pro 128 GB,"EUR 201,00",https://www.ebay.de/itm/287044752660,https://i.ebayimg.com/images/g/6YQAAeSw0U9pVUU...
313,Apple iPhone 13 Pro 256GB Graphite Grau,"EUR 222,00",https://www.ebay.de/itm/257289668732,https://i.ebayimg.com/images/g/~fUAAeSwGWlpTqB...
314,I Phone 13 Pro 256GB Ohne Simlock Set,None,https://www.ebay.de/itm/227161454953,https://i.ebayimg.com/images/g/Gp8AAeSwPSVpW8b...


In [92]:
# Define forbidden terms
forbidden_terms = [
    'refurbished',
    'iphone 12',
    'parts',
    'damaged',
    'locked',
    'pro',
    'mini',
    '256 gb',
    '512 gb',
    'verizon',
    'at&t', 
    't-mobile',
    'cricket',
    'metro',
    'boost',
    'read description'
]

In [93]:
# Create a boolean mask for filtering out forbidden terms
mask = ~items_df['Title'].str.lower().str.contains(r'\b(?:' + '|'.join(forbidden_terms) + r')\b')

# Apply the mask to filter the DataFrame
filtered_df = items_df[mask]

# Reset the index of the filtered DataFrame
filtered_df = filtered_df.reset_index(drop=True)

In [94]:
type(filtered_df)

pandas.core.frame.DataFrame

In [95]:
filtered_df

,Title,Price,Link,Image Link
0,Apple iPhone 15 - Pink 128 GB - ohne Simlock,"EUR 310,00",https://www.ebay.de/itm/257439157839,https://i.ebayimg.com/images/g/o0MAAeSwmFBpxRR...
1,Apple iPhone 13 A2633 - 256GB - (PRODUCT)RED,"EUR 210,00",https://www.ebay.de/itm/188234268200,https://i.ebayimg.com/images/g/fOMAAeSw0ZJpzXe...
2,Apple iPhone 13 - 128GB - Mitternacht (Ohne Si...,"EUR 205,99",https://www.ebay.de/itm/147228718574,https://i.ebayimg.com/images/g/~WsAAeSwiSJpyC5...
3,Apple iPhone 13 A2633 (CDMA | GSM) - 512GB - R...,"EUR 221,22",https://www.ebay.de/itm/178019284743,https://i.ebayimg.com/images/g/uMYAAeSwnuZpz~L...
4,iPhone 13 256GB Weiß Ohne Simlock 89% Akku Top...,"EUR 211,01",https://www.ebay.de/itm/358397432211,https://i.ebayimg.com/images/g/FCwAAeSwxbxpzOY...
...,...,...,...,...
66,Apple iPhone 13 128GB ohne Simlock,"EUR 214,81",https://www.ebay.de/itm/168040939731,https://i.ebayimg.com/images/g/9EUAAeSwJfVpUmg...
67,"Apple iPhone 13 256GB Blau ohne Simlock, gebra...","EUR 212,00",https://www.ebay.de/itm/297879968495,https://i.ebayimg.com/images/g/HngAAeSwp3ZpTtb...
68,Apple iPhone 13 128GB Midnight ohne Simlock,"EUR 200,00",https://www.ebay.de/itm/389446097889,https://i.ebayimg.com/images/g/HUcAAeSwAStpU3P...
69,Apple iPhone 13 128GB Midnight Black Entsperrt...,"EUR 273,63",https://www.ebay.de/itm/389454171333,https://i.ebayimg.com/images/g/8VkAAeSwistoseH...


In [96]:
# Save the filtered DataFrame to a CSV file
filtered_df.to_csv('iphone 13 128.csv', index=False)

In [97]:
#convert DataFrame to a dictionary to be able to pass it to the LLM
listing_rows = filtered_df.to_dict(orient="records")

In [98]:
#load the openai api key
load_dotenv(override=True)

api_key = os.getenv('OPENAI_API_KEY') #if fails returns None

if not api_key:
    print("No API keys were found - please head over to the .env text file to fix this!")
elif api_key.strip() != api_key:
    print("An API key waas found, please head over to the .env file and check the key is correct")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [99]:
#create an openai client
client = OpenAI()

In [100]:

#define the system prompt
system_prompt = """
You are an expert in refurbishing iphones.
You specialize in identifying profitable opportunities from eBay listings for professionals who repair and flip iPhones for a profit.

Your job is to analyze listing data and provide actionable insights for buying decisions.

Focus on:
- identifying undervalued listings
- spotting repair opportunities (e.g. cracked screen, locked, parts)
- estimating typical market value
- detecting overpriced listings
- explaining price differences based on title signals (storage, condition, model, damage, accessories)

Rules:
- Only use the provided data
- Do not invent specifications
- Be practical and business-oriented
- Highlight uncertainty when data is inconsistent
- Prioritize profitability and resale potential
- Keep the response concise and structured
- Do not use markdown headings, bold text, or decorative separators.

"""

In [101]:
user_prompt = f"""
Analyze these eBay iPhone listings for a repair-and-flip business.

Summarize:
- the typical price range
- the top 3 potential deals
- any overpriced listings
- possible reasons for price differences

Listings:
{listing_rows}
"""

In [102]:
messages = [
    {"role":"system", "content":system_prompt},
    {"role":"user", "content":user_prompt},
]

In [103]:
response = client.chat.completions.create(model = "gpt-5.4-nano", messages = messages)

In [104]:
output = response.choices[0].message.content.strip()

In [105]:
print(output)

Typical price range (based only on the provided listings)
1) iPhone 13 (mostly 128GB, ohne Simlock / unlocked)
- Common cluster: ~EUR 200 to EUR 230
- Higher outliers with “OVP/original packaging”, very high battery %, or 256GB: ~EUR 230 to ~EUR 270
- Some listings explicitly indicate batteries below typical: down to ~EUR 201–212 range with “gebraucht/ohne top” signals.

2) iPhone 13 (256GB)
- Most seen: ~EUR 211 to EUR 232
- Example higher: up to ~EUR 243 (512GB listing shows higher, but 256GB ones generally sit closer to the 220s)

3) iPhone 13 (512GB / 13 mini 512GB)
- Scarcer in the data; one 512GB iPhone 13 at ~EUR 243,83 and one iPhone 13 mini 512GB at ~EUR 274.
- These appear priced only slightly above 128GB/256GB items (likely because condition/grade/unknowns aren’t consistently described).

4) iPhone 15 (128GB, ohne Simlock)
- Common: ~EUR 303 to EUR 361
- Higher “B-Ware / factory unlocked / brandneu” style listings: ~EUR 384 to ~EUR 396+
- Outliers: ~EUR 328–350 are common; ~